# Étape 5 — Prévision de la demande

**Objectif** : produire une prévision mensuelle (horizon 6 mois) pour les 250 produits, en appliquant à chaque classe le modèle le plus adapté (mémoire §3.6).

**Stratégie d'affectation** :

| Classe / profil | Modèle |
|---|---|
| A non-intermittent | LSTM-like (MLP séquentiel) |
| B | LightGBM (lags + features temporelles + flag obsolescence) |
| C non-intermittent | SARIMA auto_arima (s=12) |
| Intermittent (Z ou ≤30 % de mois actifs) | Croston SBA |
| Obsolète (Étape 4) | Prévision = 0 |
| Cold-start (<6 mois d'historique) | Analogie famille |

**Évaluation** : MAE / RMSE / MAPE sur la fenêtre test (avril-juillet 2025).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')

## 1. Chargement des sorties du pipeline

In [ ]:
previsions = pd.read_csv(ROOT/'outputs/tables/previsions_complet.csv', parse_dates=['date'])
metrics = pd.read_csv(ROOT/'outputs/tables/comparaison_modeles.csv')
by_class = pd.read_csv(ROOT/'outputs/tables/forecast_metrics_by_class.csv')
print(f"Prévisions produites : {len(previsions):,}".replace(',', ' '))
print(f"Produits couverts : {previsions['produit_id'].nunique()}")
print(f"Horizon de planification : {previsions['date'].min().date()} → {previsions['date'].max().date()}")

## 2. Répartition des modèles utilisés

In [ ]:
previsions[['produit_id','modele_utilise']].drop_duplicates()['modele_utilise'].value_counts()

## 3. Performance par classe ABC × modèle

In [ ]:
by_class

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sub = metrics.dropna(subset=['mae'])
sns.boxplot(data=sub, x='classe_abc', y='mae', hue='modele', order=['A','B','C'], ax=ax)
ax.set_title('MAE par classe ABC × modèle (test 4 mois)'); ax.set_ylabel('MAE (unités/mois)')
plt.tight_layout(); plt.show()

## 4. Importance des features LightGBM

In [ ]:
fi = pd.read_csv(ROOT/'outputs/tables/lgbm_feature_importance.csv')
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=fi, x='importance_moy', y='feature', hue='feature', palette='flare', ax=ax, legend=False)
ax.set_title('Importance moyenne des features LightGBM (gain)')
ax.set_xlabel('Gain moyen')
plt.tight_layout(); plt.show()

## 5. Aperçu des prévisions (10 premières lignes)

In [ ]:
previsions.head(10)

## 6. Conclusion

Le pipeline de prévision tourne en ~1-2 min sur poste standard.
- Chaque produit reçoit le modèle adapté à sa classe et son profil de demande.
- Les produits obsolètes (Étape 4) ne reçoivent aucune prévision (0).
- L'intervalle de confiance simple (±15-20 %) sera affiné par les écarts résiduels de l'évaluation test pour le tableau de bord.

Les sorties (`previsions_complet.csv`) alimentent directement l'**Étape 6 — Optimisation linéaire des commandes**.